# LLM-assisted metaphor identification: Retrieval-Augmented Generation (RAG)

This notebook reproduces the RAG experiments reported in the following paper: Fuoli, M., Huang, W., Littlemore, J., Turner, S., & Wilding, E. (2026). Metaphor identification using large language models: A comparison of RAG, prompt engineering, and fine-tuning. *Applied Corpus Linguistics*, DOI: [10.1016/j.acorp.2026.100204](https://doi.org/10.1016/j.acorp.2026.100204).

## Overview

In this study, we evaluate three large language model (LLM) approaches for identifying and labeling metaphorical expressions in a corpus of film reviews:

1. Retrieval-Augmented Generation (RAG)
2. Prompt engineering
3. Fine-tuning

This notebook provides a step-by-step guide to replicating the **RAG** experiments using OpenAI's API. RAG enhances LLM performance by retrieving annotated examples from the training corpus that are semantically similar to each test text, and inserting them into the prompt as few-shot examples. This grounds the model's output in real annotation decisions from the corpus rather than relying on the model's general knowledge alone.

## What this notebook does

The notebook walks you through the full pipeline, from data preparation to evaluation. Specifically, it:

1. Loads the dataset (`metaphor_dataset_mini.csv`) and splits it into train and test sets
2. Loads the prompts used in the study (`prompts.csv`)
3. Embeds all training texts using the OpenAI Embeddings API
4. For each test text, retrieves the most similar training examples using cosine similarity
5. Constructs a few-shot prompt by inserting the retrieved examples alongside each test text
6. Sends each prompt to the OpenAI chat completions API
7. Saves the model outputs to a CSV file
8. Computes evaluation metrics (precision, recall, accuracy, and F1 score) using `evaluation.py`
9. Saves the evaluation results to CSV files

The notebook is written for non-experts. Each section explains what it is doing before the code runs.


## Before you run anything

### Check that all required files are available

Make sure these files are in the same folder as this notebook:

- `metaphor_dataset_mini.csv`
- `prompts.csv`
- `evaluation.py`

### Add your OpenAI API key

The easiest approach is to paste your API key into the next code cell.

### Safer alternative

A safer option is to store your key locally as an environment variable called `OPENAI_API_KEY` and **not** paste it into the notebook.

That method is preferred for real projects, but it is optional here. The example code is included below as commented-out code so it will not run unless you choose to use it.

On macOS or Linux, you can set your API key in a terminal before launching Jupyter:

```bash
export OPENAI_API_KEY="your_api_key_here"
```

On Windows PowerShell:

```powershell
$env:OPENAI_API_KEY="your_api_key_here"
```

Then restart VS Code or Jupyter so the notebook can see the key.

### ⚠️ Important warnings ⚠️

- Your API key is private. Do **not** share it with anyone.
- Do **not** upload or email this notebook with your key pasted into it.
- Using the OpenAI API costs money. This notebook makes both **embedding** API calls   (to build the retrieval index) and **chat completions** API calls (for inference).
- Check that your OpenAI account has enough credit or funds before you run the notebook.
- You are responsible for any charges incurred through your API key.
- We cannot take responsibility for money spent through your OpenAI account.


In [ ]:
# -------------------------------
# OpenAI API key setup
# -------------------------------
# Paste your OpenAI API key between the quotes below.
# Example:
# API_KEY = "sk-abc123..."

API_KEY = "PASTE_YOUR_OPENAI_API_KEY_HERE"

# Quick check so the notebook stops early if no key has been added.
if API_KEY == "PASTE_YOUR_OPENAI_API_KEY_HERE":
    raise ValueError(
        "Please paste your OpenAI API key into the API_KEY variable in this cell before running the notebook."
    )
else:
    print("API key added. You are ready to continue.")

# -------------------------------
# Safer alternative (optional)
# -------------------------------
# Instead of pasting your key into the notebook, you can store it locally
# as an environment variable called OPENAI_API_KEY.
#
# Example:
#
# import os
# API_KEY = os.getenv("OPENAI_API_KEY")
#
# if not API_KEY:
#     raise ValueError(
#         "OPENAI_API_KEY was not found in your environment."
#     )
#
# This method is safer because the key stays outside the notebook file.


## 1. Install and import the required packages

The first code cell installs any Python packages that this notebook needs. 
The second cell imports them into Python so the rest of the notebook can use them.


In [ ]:
# pip install -q openai pandas numpy scikit-learn tqdm ipykernel


In [ ]:
import importlib.util
import os
import re
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

import sklearn


## 2. Set file paths and experiment settings

This cell defines:

- where the dataset and prompt files live
- which OpenAI chat model and embedding model to use
- how many training examples to retrieve per test text (`TOP_K`)
- what fraction of the data to use as the retrieval pool (training set) vs. test set
- how many times each prompt should be run
- where to save the outputs
- an optional row limit for quick testing

In our study, we ran each prompt five times on each text and then aggregated the results. While this is not strictly necessary, it improves robustness, as LLM outputs may vary slightly across runs.

**`TOP_K`** controls how many annotated examples are retrieved and inserted into the prompt. Higher values give the model more context but increase prompt length and cost. We used `TOP_K = 3` in the study.

For a quick trial run, you can set `ROW_LIMIT` to a small number such as `2` or `3`.


In [ ]:
# ---------- File paths ----------
DATASET_PATH    = Path("metaphor_dataset_mini.csv")
PROMPTS_PATH    = Path("prompts.csv")
EVALUATION_PATH = Path("evaluation.py")
OUTPUT_DIR      = Path("outputs")

# ---------- Chat model ----------
MODEL_NAME = "gpt-5-mini"

# ---------- Embedding model ----------
# Used to embed training texts and test queries for retrieval.
# text-embedding-3-small is fast and cost-effective.
EMBEDDING_MODEL = "text-embedding-3-small"

# ---------- Retrieval ----------
# Number of training examples to retrieve per test text.
TOP_K = 3

# ---------- Train / test split ----------
# Fraction of the dataset used as the retrieval pool.
# The rest is used as the test set.
TRAIN_FRACTION = 0.8
RANDOM_SEED    = 42

# ---------- Repeats ----------
# Number of times to run each prompt on each dataset row.
# Set this to 5 to reproduce the original experiment design.
NUM_REPEATS = 2

# ---------- Optional testing controls ----------
# Set ROW_LIMIT to a small number like 2 or 3 for a quick test.
# Leave as None to run the full dataset.
ROW_LIMIT = None

# ---------- API pacing ----------
# Small pause between requests to reduce the chance of rate-limit issues.
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())
print("Chat model:      ", MODEL_NAME)
print("Embedding model: ", EMBEDDING_MODEL)
print("Top-K retrieved: ", TOP_K)
print("Train fraction:  ", TRAIN_FRACTION)
print("Repeats per text:", NUM_REPEATS)


## 3. Load the dataset and prompt library

The dataset contains the texts to annotate.

Expected dataset columns:

- `textid`: a unique identifier for each text
- `plain`: the raw text sent to the model
- `metaphor_tagged_text`: the gold-standard text with `<Metaphor>` tags

The prompt file contains one or more message sequences. Each prompt strategy is identified by a `pid`, and each message in that strategy has:

- `cid`: the message order within that prompt
- `role`: for example `system`, `user`, or `assistant`
- `content`: the actual message text

For RAG, the prompts are used as the **base instruction template**. Retrieved examples are then inserted into the prompt dynamically before each API call.


In [ ]:
dataset_df = pd.read_csv(DATASET_PATH)
prompts_df = pd.read_csv(PROMPTS_PATH)

if ROW_LIMIT is not None:
    dataset_df = dataset_df.head(ROW_LIMIT).copy()

required_dataset_cols = {"textid", "plain", "metaphor_tagged_text"}
missing_dataset_cols = required_dataset_cols - set(dataset_df.columns)
if missing_dataset_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_dataset_cols}")

required_prompt_cols = {"pid", "name", "cid", "role", "content"}
missing_prompt_cols = required_prompt_cols - set(prompts_df.columns)
if missing_prompt_cols:
    raise ValueError(f"Prompt file is missing required columns: {missing_prompt_cols}")

print(f"Dataset rows loaded:      {len(dataset_df)}")
print(f"Prompt rows loaded:       {len(prompts_df)}")
print(f"Unique prompt strategies: {prompts_df['pid'].nunique()}")


### Quick preview

These cells let you inspect the data before making any API calls.


In [ ]:
dataset_df.head()


In [ ]:
prompts_df.head(10)


## 4. Split the dataset into retrieval pool and test set

RAG requires a **retrieval pool** — a set of annotated examples that the system can draw on when constructing prompts. At test time, for each unseen text, the system searches this pool to find the most similar annotated examples and includes them in the prompt.

We split the dataset into:

- **Retrieval pool (training set):** annotated examples that will be embedded and indexed.
- **Test set:** texts the system has not seen, evaluated against the gold standard.

The split is controlled by `TRAIN_FRACTION` and `RANDOM_SEED` in section 2. A random seed ensures the split is reproducible across runs.


In [ ]:
train_df = dataset_df.sample(frac=TRAIN_FRACTION, random_state=RANDOM_SEED)
test_df  = dataset_df.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

if len(train_df) < TOP_K:
    raise ValueError(
        f"Retrieval pool has only {len(train_df)} examples but TOP_K={TOP_K}. "
        "Increase TRAIN_FRACTION or reduce TOP_K."
    )

print(f"Retrieval pool size: {len(train_df)}")
print(f"Test set size:       {len(test_df)}")


## 5. Prepare the prompt strategies

The prompts in `prompts.csv` are stored in a tabular format, where each row corresponds to part of a prompt (e.g. system instructions, user input, etc.).

Before we can send these prompts to the OpenAI API, we need to convert them into the structured format expected by the model.

Specifically, the API requires prompts to be formatted as a list of messages, where each message includes a role (e.g. `"system"`, `"user"`, or `"assistant"`) and its corresponding content.

In this section, we:

- Load the prompts from the CSV file
- Group and organise them by prompt ID
- Convert each prompt into the message structure required by the API

This step ensures that each prompt strategy is correctly interpreted by the model during inference.


In [ ]:
def load_prompt_strategies(prompt_table: pd.DataFrame):
    prompt_messages_by_pid = {}
    prompt_names_by_pid = {}

    for pid, group in prompt_table.sort_values(["pid", "cid"]).groupby("pid", sort=False):
        group = group.sort_values("cid").reset_index(drop=True)

        messages = []
        for _, row in group.iterrows():
            messages.append(
                {
                    "role": row["role"],
                    "content": row["content"],
                }
            )

        prompt_messages_by_pid[pid] = messages
        prompt_names_by_pid[pid] = group.loc[0, "name"]

    return prompt_messages_by_pid, prompt_names_by_pid


prompt_messages_by_pid, prompt_names_by_pid = load_prompt_strategies(prompts_df)

print("Loaded prompt ids:   ", list(prompt_messages_by_pid.keys()))
print("Example prompt name:", prompt_names_by_pid[list(prompt_names_by_pid.keys())[0]])


## 6. Create the OpenAI client

This cell creates the client that will talk to the OpenAI API. Think of the client as a messenger between this notebook and OpenAI. It is used for both the **embeddings** API (to index and retrieve examples) and the **chat completions** API (to run inference).

It uses the `API_KEY` value from the cell near the top of the notebook.


In [ ]:
client = OpenAI(api_key=API_KEY)
print("OpenAI client created successfully.")


Or if you are using a local model host (e.g. ollama) / remote non-OpenAI model host that is described as "openai compatible", you may create the client with your customised base URL and headers. Here is an example for the OpenRouter API.


In [ ]:
# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=API_KEY,
#     default_headers={
#         "HTTP-Referer": "http://localhost:5000",  # Required by OpenRouter
#         "X-Title": "MetaphorIdentification",       # Optional — your app name
#     }
# )

# print("OpenAI client created successfully.")


For a local model e.g. ollama, this is an example:


In [ ]:
# client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama",  # Required by the SDK, but Ollama ignores it
# )


Please note: when choosing a non-OpenAI model, you need to change `MODEL_NAME` and `EMBEDDING_MODEL` accordingly, and verify that the host supports the embeddings endpoint.


## 7. Embed the retrieval pool

To retrieve similar examples at test time, we first need to convert every training text into a **vector embedding** — a list of numbers that captures its semantic meaning. Two texts with similar meaning will produce embeddings that are close together in vector space.

We use the OpenAI Embeddings API (`text-embedding-3-small` by default). The embeddings are computed in a single batched API call for efficiency and then stored in a NumPy matrix for fast similarity lookups.

The embeddings are also saved to `outputs/train_embeddings.npy` so you can reload them later without repeating the API call.


In [ ]:
def get_embeddings_batch(texts: list[str], client, model: str) -> np.ndarray:
    """
    Embed a list of texts in a single API call.
    Returns a 2-D NumPy array of shape (len(texts), embedding_dim).
    """
    response = client.embeddings.create(
        model=model,
        input=texts,
    )
    vectors = [item.embedding for item in sorted(response.data, key=lambda x: x.index)]
    return np.array(vectors, dtype=np.float32)


print(f"Embedding {len(train_df)} training texts with model '{EMBEDDING_MODEL}'...")
train_embeddings = get_embeddings_batch(
    texts=train_df["plain"].tolist(),
    client=client,
    model=EMBEDDING_MODEL,
)

# Normalise rows to unit length so cosine similarity reduces to a dot product.
norms = np.linalg.norm(train_embeddings, axis=1, keepdims=True)
train_embeddings_normed = train_embeddings / np.where(norms == 0, 1, norms)

embeddings_path = OUTPUT_DIR / "train_embeddings.npy"
np.save(embeddings_path, train_embeddings_normed)

print(f"Embeddings shape: {train_embeddings_normed.shape}")
print(f"Saved to: {embeddings_path.resolve()}")


## 8. Helper functions for retrieval and prompt construction

These helper functions handle the retrieval and prompt-building steps.

They:

- embed a query text and compute cosine similarity against all training embeddings
- select the `TOP_K` most similar training examples
- insert those examples as few-shot demonstrations into the base prompt
  (as `user`/`assistant` turn pairs before the final test query)
- send the resulting prompt to the OpenAI API
- collect the model output and record token usage and cost

### How the few-shot examples are inserted

For each base prompt strategy, the retrieved examples are injected as additional `user`/`assistant` message pairs inserted just before the final user message containing the test text. This mirrors the standard few-shot chat format:

```
system: <task instructions>
user:   <example 1 plain text>
assistant: <example 1 gold tagged text>
user:   <example 2 plain text>
assistant: <example 2 gold tagged text>
...
user:   <test text>   ← the actual query
```

The pricing table below uses the official OpenAI API pricing pages for the listed text models. If OpenAI changes its pricing in the future, update the table before re-running cost calculations.


In [ ]:
# Official API prices per 1M tokens for selected text models.
# Update these values if OpenAI changes its pricing in the future.
MODEL_PRICING_PER_1M_TOKENS = {
    # --- Core GPT-5 family ---
    "gpt-5":         {"input": 1.25,  "output": 10.00},
    "gpt-5-mini":    {"input": 0.25,  "output":  2.00},
    "gpt-5-nano":    {"input": 0.05,  "output":  0.40},

    # --- GPT-5.1 series ---
    "gpt-5.1":       {"input": 1.25,  "output": 10.00},
    "gpt-5.1-mini":  {"input": 0.25,  "output":  2.00},

    # --- GPT-5.2 series ---
    "gpt-5.2":       {"input": 1.75,  "output": 14.00},
    "gpt-5.2-mini":  {"input": 0.25,  "output":  2.00},

    # --- GPT-5.3 series ---
    "gpt-5.3":       {"input": 1.75,  "output": 14.00},

    # --- GPT-5.4 (latest flagship) ---
    "gpt-5.4":       {"input": 2.50,  "output": 15.00},
    "gpt-5.4-mini":  {"input": 0.75,  "output":  4.50},
    "gpt-5.4-nano":  {"input": 0.20,  "output":  1.00},

    # --- Chat variants ---
    "gpt-5-chat":    {"input": 1.25,  "output": 10.00},
    "gpt-5.1-chat":  {"input": 1.25,  "output": 10.00},
    "gpt-5.2-chat":  {"input": 1.75,  "output": 14.00},
    "gpt-5.3-chat":  {"input": 1.75,  "output": 14.00},
}

MODEL_PRICE_ALIASES = {
    "gpt-5":             "gpt-5",
    "gpt-5-mini":        "gpt-5-mini",
    "gpt-5-nano":        "gpt-5-nano",
    "gpt-5-2025-08-01":  "gpt-5",
    "gpt-5.1":           "gpt-5.1",
    "gpt-5.1-2025-09-01": "gpt-5.1",
    "gpt-5.2":           "gpt-5.2",
    "gpt-5.2-2025-12-11": "gpt-5.2",
    "gpt-5.3":           "gpt-5.3",
    "gpt-5.4":           "gpt-5.4",
    "gpt-5.4-mini":      "gpt-5.4-mini",
    "gpt-5.4-nano":      "gpt-5.4-nano",
    "gpt-5-chat-latest":   "gpt-5-chat",
    "gpt-5.1-chat-latest": "gpt-5.1-chat",
    "gpt-5.2-chat-latest": "gpt-5.2-chat",
    "gpt-5.3-chat-latest": "gpt-5.3-chat",
}


def normalise_model_name_for_pricing(model_name: str) -> str:
    return MODEL_PRICE_ALIASES.get(model_name, model_name)


def estimate_request_cost_usd(
    prompt_tokens: int | float | None,
    completion_tokens: int | float | None,
    model_name: str,
) -> float:
    """
    Estimate request cost in USD from token counts and the model name.
    Returns NaN if the model is not in the pricing table.
    """
    pricing_model_name = normalise_model_name_for_pricing(model_name)
    pricing = MODEL_PRICING_PER_1M_TOKENS.get(pricing_model_name)

    if pricing is None:
        return np.nan

    prompt_tokens     = 0 if pd.isna(prompt_tokens)     else prompt_tokens
    completion_tokens = 0 if pd.isna(completion_tokens) else completion_tokens

    input_cost  = (prompt_tokens     / 1_000_000) * pricing["input"]
    output_cost = (completion_tokens / 1_000_000) * pricing["output"]
    return float(input_cost + output_cost)


def retrieve_top_k(
    query_text: str,
    train_pool: pd.DataFrame,
    train_embeddings_normed: np.ndarray,
    client,
    embedding_model: str,
    top_k: int,
) -> pd.DataFrame:
    """
    Embed the query text and return the top_k most similar rows from train_pool.
    """
    response = client.embeddings.create(model=embedding_model, input=[query_text])
    query_vec = np.array(response.data[0].embedding, dtype=np.float32)
    query_vec /= np.linalg.norm(query_vec) if np.linalg.norm(query_vec) > 0 else 1.0

    similarities = train_embeddings_normed @ query_vec
    top_indices  = np.argsort(similarities)[::-1][:top_k]
    return train_pool.iloc[top_indices].reset_index(drop=True)


def build_rag_messages(
    base_messages: list[dict],
    retrieved_rows: pd.DataFrame,
    test_plain_text: str,
    placeholder: str = "[#TEST_TEXT]",
) -> list[dict]:
    """
    Build a few-shot RAG prompt by:
    1. Taking all messages from base_messages up to (but not including) the final user turn.
    2. Inserting one user/assistant pair for each retrieved example.
    3. Appending the final user message with the test text substituted in.

    If no [#TEST_TEXT] placeholder is found, the test text is appended to the last user message.
    """
    messages = deepcopy(base_messages)

    # Find the last user message index — this will become the final query.
    user_indices = [i for i, m in enumerate(messages) if m["role"] == "user"]
    if not user_indices:
        raise ValueError("Base prompt contains no user message.")
    last_user_idx = user_indices[-1]

    # Substitute the test text into the final user message.
    final_user_msg = deepcopy(messages[last_user_idx])
    if placeholder in final_user_msg["content"]:
        final_user_msg["content"] = final_user_msg["content"].replace(placeholder, test_plain_text)
    else:
        final_user_msg["content"] = final_user_msg["content"].rstrip() + "\n\n" + test_plain_text

    # Build prefix: everything before the final user message.
    prefix = messages[:last_user_idx]

    # Build few-shot block: one user/assistant pair per retrieved example.
    few_shot_turns = []
    for _, ex_row in retrieved_rows.iterrows():
        few_shot_turns.append({"role": "user",      "content": ex_row["plain"]})
        few_shot_turns.append({"role": "assistant", "content": ex_row["metaphor_tagged_text"]})

    return prefix + few_shot_turns + [final_user_msg]


def call_openai_chat(client, model_name: str, messages: list[dict], temperature: float = 0.0):
    """
    Send one chat-style request and return a dictionary with the model text and token usage.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
    )

    output_text = response.choices[0].message.content
    output_text = output_text.strip() if isinstance(output_text, str) else output_text

    usage             = getattr(response, "usage", None)
    prompt_tokens     = getattr(usage, "prompt_tokens",     np.nan) if usage else np.nan
    completion_tokens = getattr(usage, "completion_tokens", np.nan) if usage else np.nan
    total_tokens      = getattr(usage, "total_tokens",      np.nan) if usage else np.nan

    return {
        "llm_output":        output_text,
        "prompt_tokens":     prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens":      total_tokens,
    }


def run_rag_on_dataset(
    dataset: pd.DataFrame,
    train_pool: pd.DataFrame,
    train_embeddings_normed: np.ndarray,
    prompt_messages_by_pid: dict,
    prompt_names_by_pid: dict,
    client,
    chat_model: str,
    embedding_model: str,
    top_k: int,
    num_repeats: int = 1,
    sleep_seconds: float = 0.2,
):
    """
    For every prompt strategy, retrieve similar examples for each test row,
    build the RAG prompt, call the API, and collect results.
    """
    rows = []

    for pid, base_messages in tqdm(prompt_messages_by_pid.items(), desc="Prompt strategies"):
        prompt_name = prompt_names_by_pid[pid]

        for repeat_number in range(1, num_repeats + 1):
            for row_idx, row in tqdm(
                dataset.iterrows(),
                total=len(dataset),
                desc=f"Samples for prompt {pid} (run {repeat_number}/{num_repeats})",
                leave=False,
            ):
                plain_text = row["plain"]

                try:
                    retrieved_rows = retrieve_top_k(
                        query_text=plain_text,
                        train_pool=train_pool,
                        train_embeddings_normed=train_embeddings_normed,
                        client=client,
                        embedding_model=embedding_model,
                        top_k=top_k,
                    )

                    rag_messages = build_rag_messages(
                        base_messages=base_messages,
                        retrieved_rows=retrieved_rows,
                        test_plain_text=plain_text,
                    )

                    response_info     = call_openai_chat(client, chat_model, rag_messages, temperature=0.0)
                    llm_output        = response_info["llm_output"]
                    prompt_tokens     = response_info["prompt_tokens"]
                    completion_tokens = response_info["completion_tokens"]
                    total_tokens      = response_info["total_tokens"]
                    estimated_cost_usd = estimate_request_cost_usd(
                        prompt_tokens, completion_tokens, chat_model
                    )
                    retrieved_textids  = ",".join(str(r) for r in retrieved_rows["textid"].tolist())
                    error_message = None
                    status        = "ok"

                except Exception as e:
                    llm_output         = None
                    prompt_tokens      = np.nan
                    completion_tokens  = np.nan
                    total_tokens       = np.nan
                    estimated_cost_usd = np.nan
                    retrieved_textids  = None
                    error_message      = str(e)
                    status             = "api_error"

                rows.append({
                    "prompt_id":               pid,
                    "prompt_name":             prompt_name,
                    "repeat_number":           repeat_number,
                    "dataset_index":           row_idx,
                    "textid":                  row["textid"],
                    "model_name":              chat_model,
                    "top_k":                   top_k,
                    "retrieved_textids":       retrieved_textids,
                    "plain":                   plain_text,
                    "gold_metaphor_tagged_text": row["metaphor_tagged_text"],
                    "llm_output":              llm_output,
                    "prompt_tokens":           prompt_tokens,
                    "completion_tokens":       completion_tokens,
                    "total_tokens":            total_tokens,
                    "estimated_cost_usd":      estimated_cost_usd,
                    "status":                  status,
                    "error_message":           error_message,
                })

                time.sleep(sleep_seconds)

    return pd.DataFrame(rows)


def summarise_experiment_costs(results_table: pd.DataFrame):
    """
    Summarise token usage and estimated cost by model.
    """
    if results_table.empty:
        return pd.DataFrame()

    summary = (
        results_table.groupby("model_name", dropna=False)
        .agg(
            num_requests        =("model_name",         "size"),
            successful_requests =("status",             lambda s: int((s == "ok").sum())),
            prompt_tokens       =("prompt_tokens",      "sum"),
            completion_tokens   =("completion_tokens",  "sum"),
            total_tokens        =("total_tokens",       "sum"),
            estimated_cost_usd  =("estimated_cost_usd", lambda s: s.sum(min_count=1)),
        )
        .reset_index()
    )

    return summary


## 9. Run the RAG pipeline on all test texts

This is the main experiment loop. For every prompt strategy and every test text, the notebook will:

1. embed the test text and retrieve the `TOP_K` most similar training examples
2. insert those examples as few-shot demonstrations into the prompt
3. send the prompt to the OpenAI API
4. save the model output
5. record token usage, the IDs of retrieved examples, and an estimated request cost
6. repeat the request as many times as specified in `NUM_REPEATS`

If you set `NUM_REPEATS = 5`, each prompt–text combination will appear in five separate rows in the output file.


In [ ]:
results_df = run_rag_on_dataset(
    dataset=test_df,
    train_pool=train_df,
    train_embeddings_normed=train_embeddings_normed,
    prompt_messages_by_pid=prompt_messages_by_pid,
    prompt_names_by_pid=prompt_names_by_pid,
    client=client,
    chat_model=MODEL_NAME,
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    num_repeats=NUM_REPEATS,
    sleep_seconds=SLEEP_BETWEEN_REQUESTS_SECONDS,
)

print(f"Total result rows: {len(results_df)}")
results_df.head()


## 10. Summarise token usage and estimated cost

This table gives a simple cost overview for the whole experiment.

It adds up:

- how many API requests were made
- how many input and output tokens were used
- the estimated total cost in US dollars

The estimates come from the pricing table defined earlier in the notebook. If you change model or pricing, update that table first.

**Note:** embedding API calls (used for retrieval) are not included in this cost summary, as they are billed separately and at a much lower rate. For reference, `text-embedding-3-small` costs $0.02 per 1M tokens as of early 2026.


In [ ]:
cost_summary_df = summarise_experiment_costs(results_df)
cost_summary_df


## 11. Save the raw model outputs and cost summary to CSV

This section saves the full row-by-row API results, including the retrieved example IDs, repeated runs, token usage and estimated cost.


In [ ]:
outputs_csv_path      = OUTPUT_DIR / "llm_outputs_rag.csv"
cost_summary_csv_path = OUTPUT_DIR / "experiment_cost_summary.csv"

results_df.to_csv(outputs_csv_path, index=False)
cost_summary_df.to_csv(cost_summary_csv_path, index=False)

print(f"Saved model outputs to: {outputs_csv_path.resolve()}")
print(f"Saved cost summary to:  {cost_summary_csv_path.resolve()}")


## 12. Load the evaluation function from `evaluation.py`

The next cell imports the function we need to evaluate the LLM output (`do_praf()`) against the gold standard manually annotated corpus from the Python script `evaluation.py`.


In [ ]:
spec = importlib.util.spec_from_file_location("evaluation_module", EVALUATION_PATH)
evaluation_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluation_module)

do_praf = evaluation_module.do_praf

print("Loaded do_praf() successfully from:", EVALUATION_PATH.resolve())


## 13. Score each output

The next function applies the evaluator row by row.

It produces a row-level metrics table, where each row corresponds to one prompt strategy applied to one test-set text in one repeat run.


In [ ]:
def evaluate_predictions(results_table: pd.DataFrame):
    row_metrics = []

    for _, row in results_table.iterrows():
        if pd.isna(row["llm_output"]) or row["llm_output"] is None or str(row["llm_output"]).strip() == "":
            continue

        try:
            metrics = do_praf(
                xml_true=row["gold_metaphor_tagged_text"],
                xml_pred=row["llm_output"],
                tag_name="Metaphor",
            )

            row_metrics.append({
                "prompt_id":         row["prompt_id"],
                "prompt_name":       row["prompt_name"],
                "repeat_number":     row["repeat_number"],
                "dataset_index":     row["dataset_index"],
                "textid":            row["textid"],
                "model_name":        row["model_name"],
                "top_k":             row["top_k"],
                "status":            row["status"],
                "estimated_cost_usd": row["estimated_cost_usd"],
                "precision":         metrics["precision"],
                "recall":            metrics["recall"],
                "accuracy":          metrics["accuracy"],
                "f1":                metrics["f1"],
                "true_pred_disp":    metrics["true_pred_disp"],
                "true_align_disp":   metrics["true_align_disp"],
            })

        except Exception as e:
            row_metrics.append({
                "prompt_id":         row["prompt_id"],
                "prompt_name":       row["prompt_name"],
                "repeat_number":     row["repeat_number"],
                "dataset_index":     row["dataset_index"],
                "textid":            row["textid"],
                "model_name":        row["model_name"],
                "top_k":             row["top_k"],
                "status":            row["status"],
                "estimated_cost_usd": row["estimated_cost_usd"],
                "precision":         np.nan,
                "recall":            np.nan,
                "accuracy":          np.nan,
                "f1":                np.nan,
                "true_pred_disp":    np.nan,
                "true_align_disp":   np.nan,
                "evaluation_error":  str(e),
            })

    return pd.DataFrame(row_metrics)


row_level_metrics_df = evaluate_predictions(results_df)
print(f"Scored rows: {len(row_level_metrics_df)}")
row_level_metrics_df.head()


## 14. Find outputs where the model did not produce any `<Metaphor>` tags

In some cases, the evaluation step may produce the following warning:

*UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples.*

### Why does this happen?

This warning occurs when the model does **not predict any metaphors at all** for a given text.

Evaluation metrics like precision are calculated based on the number of predicted positives. If the model predicts zero metaphors, then:

- there are no positive predictions
- the precision formula involves dividing by zero
- the metric becomes undefined

To handle this, the evaluation library sets precision to **0.0** for that case and raises this warning.

This is not a bug — it simply indicates that the model was **too conservative** and failed to identify any metaphors in that example.

### What this section does

This section looks for output rows where the model did **not produce any `<Metaphor>...</Metaphor>` tags** and saves these into a separate output file (`lines_without_metaphor_tags.csv`).

These cases are important to inspect because they often explain:
- drops in **recall** (missed metaphors)
- drops in **precision** (imbalanced predictions across classes)

By identifying these examples, you can better understand whether the retrieved examples are helping or whether certain prompt strategies fail to elicit tagging behaviour.


In [ ]:
METAPHOR_TAG_PATTERN = r"<Metaphor>.*?</Metaphor>"

no_metaphor_mask = (
    results_df["status"].eq("ok")
    & results_df["llm_output"].apply(
        lambda x: not bool(re.search(METAPHOR_TAG_PATTERN, str(x), re.DOTALL))
    )
)

missing_metaphor_tags_df = results_df[no_metaphor_mask].copy()

missing_metaphor_summary_df = missing_metaphor_tags_df[[
    "prompt_id",
    "prompt_name",
    "repeat_number",
    "dataset_index",
    "textid",
    "model_name",
    "plain",
    "status",
    "error_message",
]].copy()

missing_metaphor_summary_df.insert(0, "csv_row_index", missing_metaphor_tags_df.index)

missing_metaphor_csv_path = OUTPUT_DIR / "lines_without_metaphor_tags.csv"
missing_metaphor_summary_df.to_csv(missing_metaphor_csv_path, index=False)

print(f"Saved rows without metaphor tags to: {missing_metaphor_csv_path.resolve()}")
print(f"Rows with no <Metaphor>...</Metaphor> tag pair: {len(missing_metaphor_summary_df)}")
missing_metaphor_summary_df.head(20)


## 15. Aggregate the metrics by prompt strategy

This summary table gives one average score per prompt strategy across all test texts and all repeats.

It also records how many retrieved examples were used (`top_k`), how many repeats were completed, and the total estimated cost for each prompt.


In [ ]:
prompt_level_metrics_df = (
    row_level_metrics_df.groupby(["prompt_id", "prompt_name", "model_name", "top_k"], dropna=False)
    .agg(
        runs_scored          =("prompt_id",           "size"),
        unique_texts         =("textid",              "nunique"),
        repeats_completed    =("repeat_number",       "nunique"),
        avg_precision        =("precision",           "mean"),
        avg_recall           =("recall",              "mean"),
        avg_accuracy         =("accuracy",            "mean"),
        avg_f1               =("f1",                  "mean"),
        total_estimated_cost_usd=("estimated_cost_usd", lambda s: s.sum(min_count=1)),
    )
    .reset_index()
    .sort_values(["avg_f1", "avg_precision", "avg_recall"], ascending=False)
)

prompt_level_metrics_df


## 16. Save the evaluation files to CSV

This section saves the results of the evaluation in two files:

- `row_level_metrics.csv`: one row per prompt × test text × repeat with individual metric scores
- `prompt_level_metrics.csv`: one summary row per prompt strategy


In [ ]:
row_metrics_csv_path    = OUTPUT_DIR / "row_level_metrics.csv"
prompt_metrics_csv_path = OUTPUT_DIR / "prompt_level_metrics.csv"

row_level_metrics_df.to_csv(row_metrics_csv_path, index=False)
prompt_level_metrics_df.to_csv(prompt_metrics_csv_path, index=False)

print(f"Saved row-level metrics to:    {row_metrics_csv_path.resolve()}")
print(f"Saved prompt-level metrics to: {prompt_metrics_csv_path.resolve()}")


## 17. Inspect a few example predictions

This final table lets you compare:

- the original plain text
- the IDs of the training examples that were retrieved
- the gold tagged text
- the model output
- the evaluation scores

By default, the cell shows the top 10 rows of the results. Change the number in `.head(10)` to see more lines of output.


In [ ]:
example_view_df = (
    results_df.merge(
        row_level_metrics_df,
        on=["prompt_id", "prompt_name", "repeat_number", "dataset_index", "textid", "model_name"],
        how="left",
        suffixes=("", "_metric"),
    )
)

example_view_df[[
    "prompt_id",
    "prompt_name",
    "repeat_number",
    "textid",
    "retrieved_textids",
    "plain",
    "gold_metaphor_tagged_text",
    "llm_output",
    "precision",
    "recall",
    "accuracy",
    "f1",
    "estimated_cost_usd",
]].head(10)


## What the output files mean

After the notebook finishes, you should have these files in the `outputs` folder:

- `train_embeddings.npy` 
  The normalised embedding matrix for the retrieval pool.   Shape: `(n_train, embedding_dim)`. Reload with `np.load()` to skip re-embedding.

- `llm_outputs_rag.csv` 
  One row per prompt × test text × repeat. Includes the IDs of retrieved examples,   the raw model output, token usage, cost estimate and any API error message.

- `experiment_cost_summary.csv` 
  A compact summary of the number of chat completions requests, tokens used and   estimated total cost by model.

- `lines_without_metaphor_tags.csv` 
  A list of rows where the model output did not contain any `<Metaphor>...</Metaphor>` tag pair.

- `row_level_metrics.csv` 
  One row per successfully evaluated prompt × test text × repeat with precision,   recall, accuracy and F1.

- `prompt_level_metrics.csv` 
  One summary row per prompt strategy, averaging the evaluation metrics across all   scored rows and summing the estimated cost.
